In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import matplotlib.pyplot as plt

from jax import random, jit, value_and_grad, random

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error
from params import get_params
import config_utils 
from sklearn.preprocessing import StandardScaler

%matplotlib inline
# Getting the training and the test task from a LHS sampler
args = get_params()

SEED = args['seed']

import _pickle as pickle
# Add random seed for reproducibility of jnp and np results
np.random.seed(SEED)

import os, random as pyrandom
pyrandom.seed(SEED)
np.random.seed(SEED)

import jax
from jax import random as jrandom
key_master = jrandom.PRNGKey(SEED)

In [2]:
# -----------------------------
# Config
# -----------------------------
# Ts = 0.02
Ts = 0.02
SAMPLING_TIME = Ts
map_idx = 0
MAPS = ['YasMarina', 'Shanghai', 'Sepang', 'Austin', 'BrandsHatch', 'IMS', 'YasMarina_10489']
map_name = MAPS[map_idx]
VARIDX = 6  # 0: x, 1: y, 2: delta, 3: vel, 4: psi, 5: omega, 6: beta


In [3]:
import numpy as np

def lhs_1d(n, seed=0):
    rng = np.random.default_rng(seed)
    # one sample per stratum
    u = (np.arange(n) + rng.random(n)) / n
    rng.shuffle(u)  # optional in 1D, harmless
    return u

def sample_mu_lhs(mu_min, mu_max, K, anchors=(1.0,), seed=0, warp_power=None):
    anchors = [a for a in anchors if (a is not None and mu_min <= a <= mu_max)]
    anchors = sorted(set(anchors + [mu_min, mu_max]))

    n = K - len(anchors)
    if n < 0:
        raise ValueError(f"K={K} is too small for anchors={anchors}. Increase K or remove anchors.")

    u = lhs_1d(n, seed=seed)
    if warp_power is not None:
        u = u ** warp_power  # bias toward mu_min

    mus = mu_min + u * (mu_max - mu_min)
    mus = np.concatenate([np.array(anchors, dtype=float), mus])
    mus = np.unique(np.round(mus, 2))  # 👈 2 decimal limit
    return np.sort(mus)


# Example
mu_vals = sample_mu_lhs(mu_min=0.5, mu_max=1.0, K=3, anchors=(1.0,), seed=42, warp_power=2)
print(mu_vals)

[0.5 0.8 1. ]


In [4]:


# (1) Always include the lower bound as one new point
p1 = args['bounds'][0][1]  # 0.3
# (2) Get one more point via your LHS function (1-D, 1 sample)

if args['n_initial_training_envs'] > 1:
    if args['initial_training_configurations'] == 'LHS':
        p2 = config_utils.latin_hypercube_sampling_maxi_min(
            n_tasks=args['n_initial_training_envs']-1, # since we have already included one point 
            dim=args['observed_config_space_dim'],
            observed_configuration_space_interval=args['bounds']).ravel()
    elif args['initial_training_configurations'] == 'UNI':
        p2 = config_utils.sample_from_multidim_interval_uniformly(
            bounds=args['bounds'], 
            size=args['n_initial_training_envs']-1).ravel()
else:
    p2 = np.array([])

train_task_config = np.round(np.concatenate((np.array([p1]), p2.ravel()), axis=0),args['config_space_decimals'])
test_task_configurations = config_utils.create_configurations_grid(args['test_bounds'], args['test_task_count'], args['config_space_decimals']) 

# Remove duplicates between train and test tasks
train_task_config = np.unique(train_task_config)
test_task_configurations = np.unique(test_task_configurations)
test_task_configurations = test_task_configurations[~np.isin(test_task_configurations, train_task_config)]
test_task_count = len(test_task_configurations)
args['test_task_count'] = test_task_count

train_task_config = np.sort(train_task_config)[::-1]
test_task_configurations = np.sort(test_task_configurations)[::-1]
print(train_task_config, test_task_configurations)# Sort in descending order so that higher friction tasks are learned first

if SAMPLING_TIME == 0.05:
    train_task_config = np.array(mu_vals)
    train_task_config = np.sort(train_task_config)[::-1]

[1.0489 0.8642 0.4947] [0.38]


In [5]:
train_task_config.mean(), train_task_config.std()

(np.float64(0.8026), np.float64(0.23040591716938752))

In [6]:
rng = np.random.RandomState(SEED)

xscalers = {}
yscalers = {}

for task in train_task_config:
    if map_name == 'YasMarina':
        if task == 1.0489:
            xscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}/xscaler.pkl', 'rb'))
            yscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}/yscaler.pkl', 'rb'))
        else:
            xscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/xscaler.pkl', 'rb'))
            yscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/yscaler.pkl', 'rb'))
    else:
        xscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/xscaler.pkl', 'rb'))
        yscaler = pickle.load(open(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/yscaler.pkl', 'rb'))

    xscalers[f'{task}'] = xscaler
    yscalers[f'{task}'] = yscaler

In [7]:
X_dict = {} # Train set
y_dict = {}
task_ids_dict = {}

X_test_dict = {} # Test set
y_test_dict = {}
task_ids_test_dict = {}

X_val_dict = {} # Validation set
y_val_dict = {}
task_ids_val_dict = {}

all_data_X = []
all_data_y = []

C_ALPHA_COL = 6  # Column index of C_alpha feature to remove

idx = 0
for task in train_task_config:

    print(task)
    # Loading the data splits
    if map_name == 'YasMarina':
        if task == 1.0489:
            splits = np.load(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}/splits.npz')
        else:
            splits = np.load(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/splits.npz')
    elif map_name == 'Sepang':
        splits = np.load(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_2000_samples_train/splits.npz')
    else:
        splits = np.load(f'../../data/splits/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{task}/splits.npz')

    X_train = xscalers[f'{task}'].inverse_transform(splits['X_train'])
    y_train = yscalers[f'{task}'].inverse_transform(splits['y_train'])
    if splits['X_test'].shape[0]>0:  
        X_test = xscalers[f'{task}'].inverse_transform(splits['X_test'])
        y_test = yscalers[f'{task}'].inverse_transform(splits['y_test'])
    else:
        X_test = splits['X_test']
        y_test = splits['y_test']
    X_val = xscalers[f'{task}'].inverse_transform(splits['X_val'])
    y_val = yscalers[f'{task}'].inverse_transform(splits['y_val'])

    # Remove C_alpha feature (column 6)
    X_train = np.delete(X_train, C_ALPHA_COL, axis=1)
    if X_test.shape[0] > 0:
        X_test = np.delete(X_test, C_ALPHA_COL, axis=1)
    X_val = np.delete(X_val, C_ALPHA_COL, axis=1)

    X = jnp.array(X_train)
    y = jnp.array(y_train)
    f_true = y_test

    X_dict[f'{task}'] = X
    y_dict[f'{task}'] = y

    X_test_dict[f'{task}'] = X_test
    y_test_dict[f'{task}'] = y_test

    X_val_dict[f'{task}'] = X_val
    y_val_dict[f'{task}'] = y_val

    train_task_ids = np.ones(X.shape[0]) * idx
    test_task_ids = np.ones(X_test.shape[0]) * idx
    val_task_ids = np.ones(X_val.shape[0]) * idx
    task_ids_dict[f'{task}'] = train_task_ids 
    task_ids_test_dict[f'{task}'] = test_task_ids
    task_ids_val_dict[f'{task}'] = val_task_ids
    
    all_data_X_temp = np.vstack([X_train, X_val, X_test])
    all_data_y_temp = np.vstack([y_train, y_val, y_test])

    all_data_X.append(all_data_X_temp)
    all_data_y.append(all_data_y_temp)

    idx+=1

all_data_X = np.vstack(all_data_X)
all_data_y = np.vstack(all_data_y)

# Create scaler on all data
xscaler_all = StandardScaler()
yscaler_all = StandardScaler()
xscaler_all.fit(all_data_X)
yscaler_all.fit(all_data_y)
print(f"Done loading data. X shape: {all_data_X.shape} (C_alpha removed)")


1.0489
0.8642
0.4947
Done loading data. X shape: (7705, 6) (C_alpha removed)


In [8]:
X_dict.keys()

dict_keys(['1.0489', '0.8642', '0.4947'])

In [9]:
total = sum(
    X_dict[k].shape[0]
    + X_val_dict[k].shape[0]
    + X_test_dict[k].shape[0]
    for k in X_dict.keys()
)
all_data_X.shape, total

((7705, 6), 7705)

In [10]:
# Now scale the data
for task in train_task_config:
    X_dict[f'{task}'] = xscaler_all.transform(X_dict[f'{task}'])
    y_dict[f'{task}'] = yscaler_all.transform(y_dict[f'{task}'])

    if X_test_dict[f'{task}'].shape[0]>0:
        X_test_dict[f'{task}'] = xscaler_all.transform(X_test_dict[f'{task}'])
        y_test_dict[f'{task}'] = yscaler_all.transform(y_test_dict[f'{task}'])
    else:
        X_test_dict[f'{task}'] = X_test_dict[f'{task}']
        y_test_dict[f'{task}'] = y_test_dict[f'{task}']

    X_val_dict[f'{task}'] = xscaler_all.transform(X_val_dict[f'{task}'])
    y_val_dict[f'{task}'] = yscaler_all.transform(y_val_dict[f'{task}'])

In [11]:
train_task_config

array([1.0489, 0.8642, 0.4947])

In [12]:
# Keep val separate for early stopping during training.
# Val+test will be merged AFTER training for final evaluation.
X_val_all = jnp.array(np.vstack([X_val_dict[f'{t}'] for t in train_task_config]))
y_val_all = jnp.array(np.vstack([y_val_dict[f'{t}'] for t in train_task_config]))
print(f"Combined val set: X={X_val_all.shape}, y={y_val_all.shape}")

Combined val set: X=(651, 6), y=(651, 1)


In [13]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (1737, 6), y_train: (1737, 1)
X_val: (217, 6), y_val: (217, 1)
X_test: (1037, 6), y_test: (1037, 1)


In [14]:
X_train

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  2.83720493e+00,  1.64946461e+00],
       [ 3.29892921e-02,  5.67440987e-02,  0.00000000e+00,
         0.00000000e+00,  3.27866530e+00,  1.50701261e+00],
       [ 6.31295443e-02,  1.22317405e-01,  1.17387829e-02,
         0.00000000e+00,  6.57533646e+00,  1.49090385e+00],
       ...,
       [-5.45695424e-07,  1.11254176e+01,  4.28003941e-03,
        -5.95303606e-02, -3.78456265e-01,  7.26745725e-02],
       [ 1.45294575e-03,  1.11178484e+01, -1.73589297e-02,
        -5.69845019e-02, -5.28636754e-01, -3.91474187e-01],
       [-6.37653799e-03,  1.11072757e+01, -2.90227011e-02,
        -5.40868386e-02, -2.23114923e-01,  1.32991374e-01]],
      shape=(1737, 6))

In [15]:
X_train_original = xscaler_all.inverse_transform(X_dict['0.8642'])

In [16]:
X_train_original

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  2.83720493,
         1.64946461],
       [ 0.03298929,  0.0567441 ,  0.        ,  0.        ,  3.2786653 ,
         1.50701261],
       [ 0.06312954,  0.1223174 ,  0.01173878,  0.        ,  6.57533646,
         1.49090385],
       ...,
       [ 0.30907395,  4.80918475,  3.55253774, -0.18369489,  1.82685757,
         0.85926336],
       [ 0.32625922,  4.8457219 ,  3.31838603, -0.19372703,  3.78739762,
        -0.62970471],
       [ 0.31366512,  4.92146985,  2.8900899 , -0.19701771, -3.77161932,
         1.17021155]], shape=(1737, 6))

## SVGP Model (Martin Krasser Implementation) — No C_alpha

Adapted from the [sparse GP blog post](http://krasserm.github.io/2020/12/12/gaussian-processes-sparse/) for 6D input (C_alpha removed).

Trains one independent SVGP per task (friction value). Each SVGP jointly optimizes kernel hyperparameters $\theta$ and inducing point locations $\mathbf{X}_m$ via L-BFGS-B.


In [17]:
# ============================================================
# SVGP Model Functions (Krasser, adapted for d-dimensional input)
# ============================================================

# --- Isotropic RBF kernel ---
def kernel_iso(X1, X2, theta):
    """Isotropic squared exponential kernel. theta = [length_scale, signal_std]."""
    sqdist = jnp.sum(X1 ** 2, 1).reshape(-1, 1) + jnp.sum(X2 ** 2, 1) - 2 * jnp.dot(X1, X2.T)
    return theta[1] ** 2 * jnp.exp(-0.5 / theta[0] ** 2 * sqdist)

def kernel_iso_diag(n, theta):
    return jnp.full(shape=n, fill_value=theta[1] ** 2)

# --- ARD RBF kernel ---
def kernel_ard(X1, X2, theta):
    """ARD squared exponential kernel. theta = [ell_0, ..., ell_{d-1}, signal_std]."""
    ells = theta[:-1]  # (d,)
    sigma = theta[-1]
    X1s = X1 / ells
    X2s = X2 / ells
    sqdist = jnp.sum(X1s ** 2, 1).reshape(-1, 1) + jnp.sum(X2s ** 2, 1) - 2 * jnp.dot(X1s, X2s.T)
    return sigma ** 2 * jnp.exp(-0.5 * sqdist)

def kernel_ard_diag(n, theta):
    return jnp.full(shape=n, fill_value=theta[-1] ** 2)

# --- Kernel selection ---
def build_kernel(use_ard, d):
    """Returns (kernel_fn, kernel_diag_fn, num_kernel_params)."""
    if use_ard:
        return kernel_ard, kernel_ard_diag, d + 1  # d length scales + 1 signal std
    else:
        return kernel_iso, kernel_iso_diag, 2       # 1 length scale + 1 signal std

# --- Utilities ---
def jitter(d, value=1e-6):
    return jnp.eye(d) * value

def softplus(X):
    return jnp.log(1 + jnp.exp(X))

def softplus_inv(X):
    return jnp.log(jnp.exp(X) - 1)

def pack_params(theta, X_m):
    """Pack kernel params and inducing locations into a flat vector."""
    return jnp.concatenate([softplus_inv(theta), X_m.ravel()])

def unpack_params(params, d, num_kernel_params):
    """Unpack flat vector into kernel params and (m, d) inducing locations."""
    theta = softplus(params[:num_kernel_params])
    X_m = params[num_kernel_params:].reshape(-1, d)
    return theta, X_m

def nlb_fn(X, y, sigma_y, d, kernel, kernel_diag, num_kernel_params):
    """
    Factory returning (value, grad) of negative lower bound for L-BFGS-B.
    """
    n = X.shape[0]

    def nlb(params):
        theta, X_m = unpack_params(params, d, num_kernel_params)
        m = X_m.shape[0]

        K_mm = kernel(X_m, X_m, theta) + jitter(m)
        K_mn = kernel(X_m, X, theta)

        L = jnp.linalg.cholesky(K_mm)
        A = jsp.linalg.solve_triangular(L, K_mn, lower=True) / sigma_y
        AAT = A @ A.T
        B = jnp.eye(m) + AAT
        LB = jnp.linalg.cholesky(B)
        c = jsp.linalg.solve_triangular(LB, A.dot(y), lower=True) / sigma_y

        lb = -n / 2 * jnp.log(2 * jnp.pi)
        lb -= jnp.sum(jnp.log(jnp.diag(LB)))
        lb -= n / 2 * jnp.log(sigma_y ** 2)
        lb -= 0.5 / sigma_y ** 2 * y.T.dot(y)
        lb += 0.5 * c.T.dot(c)
        lb -= 0.5 / sigma_y ** 2 * jnp.sum(kernel_diag(n, theta))
        lb += 0.5 * jnp.trace(AAT)

        return -lb[0, 0]

    nlb_grad = jit(value_and_grad(nlb))

    def nlb_grad_wrapper(params):
        value, grads = nlb_grad(params)
        return np.array(value), np.array(grads)

    return nlb_grad_wrapper

def phi_opt_fn(kernel):
    """Factory returning a JIT-compiled phi_opt for the given kernel."""
    @jit
    def phi_opt(theta, X_m, X, y, sigma_y):
        """Compute optimal variational parameters (closed-form)."""
        precision = 1.0 / sigma_y ** 2
        K_mm = kernel(X_m, X_m, theta) + jitter(X_m.shape[0])
        K_mm_inv = jnp.linalg.inv(K_mm)
        K_nm = kernel(X, X_m, theta)
        K_mn = K_nm.T
        Sigma = jnp.linalg.inv(K_mm + precision * K_mn @ K_nm)
        mu_m = precision * (K_mm @ Sigma @ K_mn).dot(y)
        A_m = K_mm @ Sigma @ K_mm
        return mu_m, A_m, K_mm_inv
    return phi_opt

def q_fn(kernel):
    """Factory returning a JIT-compiled predictive function for the given kernel."""
    @jit
    def q(X_test, theta, X_m, mu_m, A_m, K_mm_inv):
        """Approximate posterior predictive (mean and covariance)."""
        K_ss = kernel(X_test, X_test, theta)
        K_sm = kernel(X_test, X_m, theta)
        K_ms = K_sm.T
        f_q = (K_sm @ K_mm_inv).dot(mu_m)
        f_q_cov = K_ss - K_sm @ K_mm_inv @ K_ms + K_sm @ K_mm_inv @ A_m @ K_mm_inv @ K_ms
        return f_q, f_q_cov
    return q

print("SVGP functions defined.")

SVGP functions defined.


In [18]:
# ============================================================
# Hyperparameters
# ============================================================
svgp_hparams = {
    # Kernel
    'use_ard': True,                     # True: ARD RBF (per-dim length scales), False: isotropic RBF
    'init_length_scale': 1.0,            # initial l (optimized via L-BFGS-B)
    'init_signal_std': 1.0,              # initial sigma_f (optimized via L-BFGS-B)

    # Noise
    'sigma_y': 0.000319,      # observation noise std (fixed, not optimized)

    # Inducing points
    'num_inducing': 12,                  # m
    'inducing_init': 'kmeans',           # 'kmeans' or 'random_subset'

    # Optimization
    'optimizer': 'L-BFGS-B',
    'maxiter': 100000,

    # Early stopping (validation-based)
    'val_check_every': 1,               # check val RMSE every N L-BFGS-B iterations
    'patience': 200,                     # stop if no val improvement for this many checks

    # Data
    'input_dim': 6,                      # d (S_delta, S_vel, S_omega, S_beta, A_a, A_sv) — NO C_alpha
    'output_dim': 1,                     # single output per SVGP

    # Numerical stability
    'jitter': 1e-6,

    # Seed
    'seed': SEED,
}

print("SVGP Hyperparameters (no C_alpha):")
for k, v in svgp_hparams.items():
    print(f"  {k}: {v}")


SVGP Hyperparameters (no C_alpha):
  use_ard: True
  init_length_scale: 1.0
  init_signal_std: 1.0
  sigma_y: 0.000319
  num_inducing: 12
  inducing_init: kmeans
  optimizer: L-BFGS-B
  maxiter: 100000
  val_check_every: 1
  patience: 200
  input_dim: 6
  output_dim: 1
  jitter: 1e-06
  seed: 42


In [19]:
# ============================================================
# Concatenate all tasks into a single training set
# ============================================================
from sklearn.cluster import KMeans

m = svgp_hparams['num_inducing']
sigma_y = svgp_hparams['sigma_y']
d = svgp_hparams['input_dim']
use_ard = svgp_hparams['use_ard']

# Build kernel based on use_ard flag
kernel, kernel_diag, num_kernel_params = build_kernel(use_ard, d)
phi_opt = phi_opt_fn(kernel)
q = q_fn(kernel)
print(f"Kernel: {'ARD RBF' if use_ard else 'Isotropic RBF'} ({num_kernel_params} kernel params)")

X_train_all = jnp.array(np.vstack([X_dict[f'{t}'] for t in train_task_config]))
y_train_all = jnp.array(np.vstack([y_dict[f'{t}'] for t in train_task_config]))

print(f"Combined training set: X={X_train_all.shape}, y={y_train_all.shape}")
print(f"Combined val set:      X={X_val_all.shape}, y={y_val_all.shape}")

# ============================================================
# Train a single SVGP on all tasks (with val-based early stopping)
# ============================================================
n = X_train_all.shape[0]

# Initialize inducing points
if svgp_hparams['inducing_init'] == 'kmeans':
    kmeans = KMeans(n_clusters=m, random_state=svgp_hparams['seed'], n_init=10)
    kmeans.fit(np.array(X_train_all))
    X_m0 = jnp.array(kmeans.cluster_centers_)
    print(f"Inducing points initialized via k-means (inertia={kmeans.inertia_:.2f})")
else:
    rng_idx = np.random.RandomState(svgp_hparams['seed'])
    idx_subset = rng_idx.choice(n, size=m, replace=False)
    X_m0 = X_train_all[idx_subset]
    print("Inducing points initialized via random subset")

# Initial kernel hyperparameters
if use_ard:
    theta0 = jnp.ones(num_kernel_params)
else:
    theta0 = jnp.array([svgp_hparams['init_length_scale'],
                         svgp_hparams['init_signal_std']])

params0 = pack_params(theta0, X_m0)
print(f"Total parameters: {len(params0)} ({num_kernel_params} kernel + {m}*{d} inducing locations)")

# ---- Early stopping callback ----
val_check_every = svgp_hparams['val_check_every']
patience = svgp_hparams['patience']

es_state = {
    'iter': 0,
    'best_val_rmse': float('inf'),
    'best_params': None,
    'best_iter': 0,
    'checks_without_improvement': 0,
    'val_rmse_history': [],
    'train_nlb_history': [],
}

def early_stopping_callback(xk):
    """L-BFGS-B callback: monitor val RMSE, save best params."""
    es_state['iter'] += 1
    it = es_state['iter']

    if it % val_check_every != 0:
        return False  # don't stop

    # Compute val RMSE
    theta_cur, X_m_cur = unpack_params(xk, d, num_kernel_params)
    mu_m_cur, A_m_cur, K_mm_inv_cur = phi_opt(theta_cur, X_m_cur, X_train_all, y_train_all, sigma_y)
    f_val, _ = q(X_val_all, theta_cur, X_m_cur, mu_m_cur, A_m_cur, K_mm_inv_cur)

    # RMSE in original scale
    y_val_orig = yscaler_all.inverse_transform(np.array(y_val_all)).ravel()
    f_val_orig = yscaler_all.inverse_transform(np.array(f_val)).ravel()
    val_rmse = np.sqrt(mean_squared_error(y_val_orig, f_val_orig))

    es_state['val_rmse_history'].append((it, val_rmse))

    if val_rmse < es_state['best_val_rmse']:
        es_state['best_val_rmse'] = val_rmse
        es_state['best_params'] = np.array(xk).copy()
        es_state['best_iter'] = it
        es_state['checks_without_improvement'] = 0
    else:
        es_state['checks_without_improvement'] += 1

    if it % (val_check_every * 10) == 0:
        print(f"  iter {it:>5d} | val RMSE: {val_rmse:.6f} | best: {es_state['best_val_rmse']:.6f} (iter {es_state['best_iter']})")

    if es_state['checks_without_improvement'] >= patience:
        print(f"\n  Early stopping at iter {it}. No improvement for {patience} checks.")
        print(f"  Best val RMSE: {es_state['best_val_rmse']:.6f} at iter {es_state['best_iter']}")
        return True  # signal to stop (scipy >= 1.12)

    return False

print(f"\nEarly stopping: check every {val_check_every} iters, patience={patience} checks")

res = minimize(
    fun=nlb_fn(X_train_all, y_train_all, sigma_y, d, kernel, kernel_diag, num_kernel_params),
    x0=np.array(params0),
    method=svgp_hparams['optimizer'],
    jac=True,
    callback=early_stopping_callback,
    options={'maxiter': svgp_hparams['maxiter'], 'disp': True}
)

# ---- Use best-val params if available ----
if es_state['best_params'] is not None:
    best_x = es_state['best_params']
    print(f"\nUsing best-val params from iter {es_state['best_iter']} (val RMSE={es_state['best_val_rmse']:.6f})")
    print(f"(L-BFGS-B finished at iter {es_state['iter']}, final NLB={res.fun:.4f})")
else:
    best_x = res.x
    print(f"\nNo val improvement tracked; using final L-BFGS-B params.")

theta_opt, X_m_opt = unpack_params(best_x, d, num_kernel_params)
mu_m_opt, A_m_opt, K_mm_inv = phi_opt(theta_opt, X_m_opt, X_train_all, y_train_all, sigma_y)

if use_ard:
    print(f"\nOptimized ARD length scales: {np.array(theta_opt[:-1]).round(4)}")
    print(f"Optimized signal_std: {float(theta_opt[-1]):.4f}")
else:
    print(f"\nOptimized: length_scale={float(theta_opt[0]):.4f}, signal_std={float(theta_opt[1]):.4f}")

model = {
    'theta': theta_opt,
    'X_m': X_m_opt,
    'mu_m': mu_m_opt,
    'A_m': A_m_opt,
    'K_mm_inv': K_mm_inv,
    'res': res,
    'use_ard': use_ard,
    'best_iter': es_state['best_iter'],
    'best_val_rmse': es_state['best_val_rmse'],
}
print("\nTraining complete.")

Kernel: ARD RBF (7 kernel params)
Combined training set: X=(5211, 6), y=(5211, 1)
Combined val set:      X=(651, 6), y=(651, 1)
Inducing points initialized via k-means (inertia=9859.60)
Total parameters: 79 (7 kernel + 12*6 inducing locations)

Early stopping: check every 1 iters, patience=200 checks
  iter    10 | val RMSE: 0.011892 | best: 0.011892 (iter 10)
  iter    20 | val RMSE: 0.008613 | best: 0.008365 (iter 16)
  iter    30 | val RMSE: 0.008355 | best: 0.008182 (iter 27)
  iter    40 | val RMSE: 0.008416 | best: 0.008182 (iter 27)
  iter    50 | val RMSE: 0.008531 | best: 0.008182 (iter 27)
  iter    60 | val RMSE: 0.008373 | best: 0.008182 (iter 27)
  iter    70 | val RMSE: 0.008484 | best: 0.008182 (iter 27)
  iter    80 | val RMSE: 0.008520 | best: 0.008182 (iter 27)
  iter    90 | val RMSE: 0.008500 | best: 0.008182 (iter 27)
  iter   100 | val RMSE: 0.008378 | best: 0.008182 (iter 27)
  iter   110 | val RMSE: 0.008238 | best: 0.008182 (iter 27)
  iter   120 | val RMSE: 0.

In [20]:
# ============================================================
# Merge val+test for final evaluation, then compute RMSE
# ============================================================
X_eval_dict = {}
y_eval_dict = {}
for task in train_task_config:
    X_eval_dict[f'{task}'] = np.vstack([X_val_dict[f'{task}'], X_test_dict[f'{task}']])
    y_eval_dict[f'{task}'] = np.vstack([y_val_dict[f'{task}'], y_test_dict[f'{task}']])

Y_NAMES = ['Y_x', 'Y_y', 'Y_delta', 'Y_vel', 'Y_psi', 'Y_omega', 'Y_beta']
var_name = Y_NAMES[VARIDX]

print(f"Variable: {var_name} (VARIDX={VARIDX})")
print(f"Best val RMSE: {es_state['best_val_rmse']:.6f} at iter {es_state['best_iter']}")
print(f"\n{'Task':<10} {'Train RMSE':<15} {'Test RMSE':<15} {'Test RMSE (orig)':<18} {'Test MAE (orig)':<15}")
print("-" * 73)

all_test_pred_orig = []
all_test_true_orig = []

for task in train_task_config:
    task_key = f'{task}'

    # --- Train RMSE (scaled space) ---
    X_tr = jnp.array(X_dict[task_key])
    y_tr = np.array(y_dict[task_key]).ravel()
    f_tr, _ = q(X_tr, model['theta'], model['X_m'], model['mu_m'], model['A_m'], model['K_mm_inv'])
    f_tr = np.array(f_tr).ravel()
    rmse_train = np.sqrt(mean_squared_error(y_tr, f_tr))

    # --- Test RMSE (scaled space) ---
    X_te = jnp.array(X_eval_dict[task_key])
    y_te = np.array(y_eval_dict[task_key]).ravel()
    f_te, _ = q(X_te, model['theta'], model['X_m'], model['mu_m'], model['A_m'], model['K_mm_inv'])
    f_te = np.array(f_te).ravel()
    rmse_test = np.sqrt(mean_squared_error(y_te, f_te))

    # --- Test RMSE in original scale ---
    y_te_orig = yscaler_all.inverse_transform(y_te.reshape(-1, 1)).ravel()
    f_te_orig = yscaler_all.inverse_transform(f_te.reshape(-1, 1)).ravel()
    rmse_test_orig = np.sqrt(mean_squared_error(y_te_orig, f_te_orig))
    mae_test_orig = mean_absolute_error(y_te_orig, f_te_orig)

    all_test_pred_orig.append(f_te_orig)
    all_test_true_orig.append(y_te_orig)

    print(f"mu={task:<6} {rmse_train:<15.6f} {rmse_test:<15.6f} {rmse_test_orig:<18.6f} {mae_test_orig:<15.6f}")

# --- Overall ---
all_test_pred_orig = np.concatenate(all_test_pred_orig)
all_test_true_orig = np.concatenate(all_test_true_orig)
rmse_overall = np.sqrt(mean_squared_error(all_test_true_orig, all_test_pred_orig))
mae_overall = mean_absolute_error(all_test_true_orig, all_test_pred_orig)

print("-" * 73)
print(f"{'OVERALL':<10} {'':15} {'':15} {rmse_overall:<18.6f} {mae_overall:<15.6f}")

Variable: Y_beta (VARIDX=6)
Best val RMSE: 0.007366 at iter 1437

Task       Train RMSE      Test RMSE       Test RMSE (orig)   Test MAE (orig)
-------------------------------------------------------------------------
mu=1.0489 0.076121        0.067038        0.005742           0.004021       
mu=0.8642 0.049365        0.043917        0.003761           0.002570       
mu=0.4947 0.070928        0.106347        0.009109           0.006852       
-------------------------------------------------------------------------
OVERALL                                    0.007285           0.005056       


In [21]:
# Variable: Y_beta (VARIDX=6)
# Best val RMSE: 0.001664 at iter 13520

# Task       Train RMSE      Test RMSE       Test RMSE (orig)   Test MAE (orig)
# -------------------------------------------------------------------------
# mu=1.0489 0.029040        0.016337        0.001399           0.000951       
# mu=0.8642 0.023220        0.018329        0.001570           0.001139       
# mu=0.4947 0.016688        0.017629        0.001510           0.001055       
# -------------------------------------------------------------------------
# OVERALL                                    0.001502           0.001053

In [22]:
# Optimized kernel: length_scale=14.9266, signal_std=9.0343

# Task       Test RMSE (orig)   Test MAE (orig)
# -------------------------------------------
# mu=1.0489 0.001821           0.001279       
# mu=0.8642 0.001721           0.001273       
# mu=0.4947 0.002413           0.001525       
# -------------------------------------------
# OVERALL    0.002117           0.001401       


In [23]:
import json

def save_model_and_scalers_npz(path, model: dict, xscaler, yscaler, meta: dict = None):
    meta = {} if meta is None else dict(meta)

    def to_np(v):
        if isinstance(v, (int, float, bool, str, type(None))):
            return v
        try:
            return np.asarray(v)
        except Exception:
            return v

    arrays = {}

    # ---- model dict ----
    model_json = {}
    for k, v in model.items():
        vv = to_np(v)
        if isinstance(vv, np.ndarray):
            arrays[f"model__{k}"] = vv
        elif isinstance(vv, (int, float, bool)) or vv is None or isinstance(vv, str):
            model_json[k] = vv
        elif isinstance(vv, (list, tuple)):
            try:
                arrays[f"model__{k}"] = np.asarray(vv)
            except Exception:
                model_json[k] = vv
        else:
            try:
                json.dumps(vv)
                model_json[k] = vv
            except Exception as e:
                raise TypeError(
                    f"model[{k!r}] is not np/JSON serializable (type={type(v)}). "
                    f"Convert it to arrays/numbers/strings before saving. Error: {e}"
                )

    arrays["model_json"] = np.asarray(json.dumps(model_json), dtype=np.str_)

    # ---- scalers ----
    def pack_scaler(prefix, s):
        arrays[f"{prefix}__mean"] = np.asarray(s.mean_, dtype=np.float64)
        arrays[f"{prefix}__scale"] = np.asarray(s.scale_, dtype=np.float64)
        if hasattr(s, "var_") and s.var_ is not None:
            arrays[f"{prefix}__var"] = np.asarray(s.var_, dtype=np.float64)
        arrays[f"{prefix}__n_features_in"] = np.asarray(
            int(getattr(s, "n_features_in_", len(s.mean_))), dtype=np.int32
        )
        arrays[f"{prefix}__with_mean"] = np.asarray(bool(getattr(s, "with_mean", True)))
        arrays[f"{prefix}__with_std"] = np.asarray(bool(getattr(s, "with_std", True)))

    pack_scaler("xscaler", xscaler)
    pack_scaler("yscaler", yscaler)

    # ---- metadata ----
    arrays["meta_json"] = np.asarray(json.dumps(meta), dtype=np.str_)

    np.savez_compressed(path, **arrays)


# ---------- LOAD ----------
from sklearn.preprocessing import StandardScaler

def load_model_and_scalers_npz(path):
    data = np.load(path, allow_pickle=False)

    meta = json.loads(str(data["meta_json"]))
    model_json = json.loads(str(data["model_json"]))

    model = dict(model_json)
    for key in data.files:
        if key.startswith("model__"):
            k = key.split("__", 1)[1]
            model[k] = data[key]

    def unpack_scaler(prefix):
        s = StandardScaler(
            with_mean=bool(data[f"{prefix}__with_mean"]),
            with_std=bool(data[f"{prefix}__with_std"]),
        )
        s.mean_ = data[f"{prefix}__mean"]
        s.scale_ = data[f"{prefix}__scale"]
        if f"{prefix}__var" in data.files:
            s.var_ = data[f"{prefix}__var"]
        s.n_features_in_ = int(data[f"{prefix}__n_features_in"])
        return s

    xscaler = unpack_scaler("xscaler")
    yscaler = unpack_scaler("yscaler")

    return model, xscaler, yscaler, meta

print("Save/load functions defined.")

Save/load functions defined.


In [24]:
# ============================================================
# Save model + scalers (pkl and npz)
# ============================================================

if VARIDX == 5:
    name = 'krasser_svgp_omega'
elif VARIDX == 6:
    name = 'krasser_svgp_beta'
else:
    name = f'krasser_svgp_var{VARIDX}'

os.makedirs('../../data/models', exist_ok=True)

# --- Strip scipy OptimizeResult from model (not serializable) ---
model_save = {k: v for k, v in model.items() if k != 'res'}
model_save['nlb_final'] = float(model['res'].fun)
model_save['converged'] = bool(model['res'].success)
model_save['sigma_y'] = float(sigma_y)

# --- Pickle save ---
pkl_path = f'../../data/models/{name}_{map_name}_VAR{VARIDX}_jax_Ts{SAMPLING_TIME:.3f}_MPPI_data_ard_kernel_no_c_alpha_M_{m}.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump((model_save, xscaler_all, yscaler_all), f)
print(f"Pickle saved: {pkl_path}")

# --- NPZ save ---
npz_path = f'../../data/models/{name}_{map_name}_VAR{VARIDX}_jax_Ts{SAMPLING_TIME:.3f}_MPPI_data_ard_kernel_no_c_alpha_M_{m}.npz'
save_model_and_scalers_npz(
    npz_path,
    model=model_save,
    xscaler=xscaler_all,
    yscaler=yscaler_all,
    meta={
        "VARIDX": int(VARIDX),
        "map_name": map_name,
        "SAMPLING_TIME": float(SAMPLING_TIME),
        "num_inducing": int(svgp_hparams['num_inducing']),
        "sigma_y": float(sigma_y),
        "train_tasks": train_task_config.tolist(),
        "c_alpha_removed": True,
    }
)
print(f"NPZ saved:    {npz_path}")

# --- Verify roundtrip ---
npz_model, npz_xscaler, npz_yscaler, npz_meta = load_model_and_scalers_npz(npz_path)
print(f"\nVerification - loaded keys: {list(npz_model.keys())}")
print(f"Metadata: {npz_meta}")
print(f"theta match: {np.allclose(np.array(model_save['theta']), npz_model['theta'])}")
print(f"X_m match:   {np.allclose(np.array(model_save['X_m']), npz_model['X_m'])}")
print(f"mu_m match:  {np.allclose(np.array(model_save['mu_m']), npz_model['mu_m'])}")


Pickle saved: ./gp/data/models/krasser_svgp_beta_YasMarina_VAR6_jax_Ts0.020_MPPI_data_ard_kernel_no_c_alpha_M_12.pkl
NPZ saved:    ./gp/data/models/krasser_svgp_beta_YasMarina_VAR6_jax_Ts0.020_MPPI_data_ard_kernel_no_c_alpha_M_12.npz

Verification - loaded keys: ['use_ard', 'best_iter', 'best_val_rmse', 'nlb_final', 'converged', 'sigma_y', 'theta', 'X_m', 'mu_m', 'A_m', 'K_mm_inv']
Metadata: {'VARIDX': 6, 'map_name': 'YasMarina', 'SAMPLING_TIME': 0.02, 'num_inducing': 12, 'sigma_y': 0.000319, 'train_tasks': [1.0489, 0.8642, 0.4947], 'c_alpha_removed': True}
theta match: True
X_m match:   True
mu_m match:  True


In [25]:
# ============================================================
# Sanity check: RMSE from loaded model matches original
# ============================================================
loaded_model, loaded_xscaler, loaded_yscaler, loaded_meta = load_model_and_scalers_npz(npz_path)

# Convert loaded arrays to jnp
loaded_theta = jnp.array(loaded_model['theta'])
loaded_X_m = jnp.array(loaded_model['X_m'])
loaded_mu_m = jnp.array(loaded_model['mu_m'])
loaded_A_m = jnp.array(loaded_model['A_m'])
loaded_K_mm_inv = jnp.array(loaded_model['K_mm_inv'])

print(f"Loaded model from: {npz_path}")
print(f"Variable: {var_name} (VARIDX={VARIDX})")
print(f"\n{'Task':<10} {'Test RMSE (orig)':<18} {'Test MAE (orig)':<15}")
print("-" * 43)

all_test_pred_loaded = []
all_test_true_loaded = []

for task in train_task_config:
    task_key = f'{task}'

    X_te = jnp.array(X_eval_dict[task_key])
    y_te = np.array(y_eval_dict[task_key]).ravel()
    f_te, _ = q(X_te, loaded_theta, loaded_X_m, loaded_mu_m, loaded_A_m, loaded_K_mm_inv)
    f_te = np.array(f_te).ravel()

    y_te_orig = loaded_yscaler.inverse_transform(y_te.reshape(-1, 1)).ravel()
    f_te_orig = loaded_yscaler.inverse_transform(f_te.reshape(-1, 1)).ravel()
    rmse_orig = np.sqrt(mean_squared_error(y_te_orig, f_te_orig))
    mae_orig = mean_absolute_error(y_te_orig, f_te_orig)

    all_test_pred_loaded.append(f_te_orig)
    all_test_true_loaded.append(y_te_orig)

    print(f"mu={task:<6} {rmse_orig:<18.6f} {mae_orig:<15.6f}")

all_test_pred_loaded = np.concatenate(all_test_pred_loaded)
all_test_true_loaded = np.concatenate(all_test_true_loaded)
rmse_overall_loaded = np.sqrt(mean_squared_error(all_test_true_loaded, all_test_pred_loaded))
mae_overall_loaded = mean_absolute_error(all_test_true_loaded, all_test_pred_loaded)

print("-" * 43)
print(f"{'OVERALL':<10} {rmse_overall_loaded:<18.6f} {mae_overall_loaded:<15.6f}")
print(f"\nRMSE match: {np.isclose(rmse_overall, rmse_overall_loaded)}")

Loaded model from: ./gp/data/models/krasser_svgp_beta_YasMarina_VAR6_jax_Ts0.020_MPPI_data_ard_kernel_no_c_alpha_M_12.npz
Variable: Y_beta (VARIDX=6)

Task       Test RMSE (orig)   Test MAE (orig)
-------------------------------------------
mu=1.0489 0.005742           0.004021       
mu=0.8642 0.003761           0.002570       
mu=0.4947 0.009109           0.006852       
-------------------------------------------
OVERALL    0.007285           0.005056       

RMSE match: True
